In [1]:
%pip install torch

%pip install nltk

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 11.5 MB/s  0:00:00

   ---------------------------------------- 0/5 [tqdm]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ---------------- ----------------------- 2/5 [joblib]
   ------------------------ --------------- 3/5 [click]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   -------------------------------- ------- 4/5 [nltk]
   ----------

In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
from functools import lru_cache


@lru_cache(maxsize=None)
def get_synsets(token):
    return wn.synsets(token)

def tokenize(text):
    return word_tokenize(text, preserve_line=True)

nltk.data.path = ["C:/nltk_data"]

# Ensure required NLTK data is available; download if missing
nltk.download('punkt', download_dir='C:/nltk_data', force=True)
nltk.download('wordnet', download_dir='C:/nltk_data')
nltk.download('omw-1.4', download_dir='C:/nltk_data')

nltk.data.path.clear()
nltk.data.path.append(r"C:\nltk_data")

print(nltk.data.path)

df = pd.read_csv("train_data.csv")


LookupError: 
**********************************************************************
  Resource [93mstopwords[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('stopwords')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/stopwords[0m

  Searched in:
    - 'C:\\nltk_data'
**********************************************************************


In [51]:
print(nltk.data.path)

import os

for path in nltk.data.path:
    print(path, "exists:", os.path.exists(path))

import os

print(os.listdir("C:/nltk_data"))
print(os.listdir("C:/Users/hugov/AppData/Roaming/nltk_data"))
print(os.listdir("C:/nltk_data/tokenizers/punkt"))

['C:\\nltk_data']
C:\nltk_data exists: True
['corpora', 'tokenizers']
['corpora', 'tokenizers']
['.DS_Store', 'czech.pickle', 'danish.pickle', 'dutch.pickle', 'english.pickle', 'estonian.pickle', 'finnish.pickle', 'french.pickle', 'german.pickle', 'greek.pickle', 'italian.pickle', 'malayalam.pickle', 'norwegian.pickle', 'polish.pickle', 'portuguese.pickle', 'PY3', 'README', 'russian.pickle', 'slovene.pickle', 'spanish.pickle', 'swedish.pickle', 'turkish.pickle']


# FIRST MODEL ATTEMPT

This model uses ARGMIN word similartiy because a sentence that correctly explains why a statement is wrong uses new words aka less similair words, I had one run wiht 0.8 and then dropped to 0.2, 

I tested it on N = 5, its not ready for large scale trainer

EVAL is just done by acc% = total - wrong 

Somehow my accuracy dropped to 20% :(



# NOTES for next time

- REMOVE STOPWORDS
- WE USE WUP for similairity but that doesnt mean its the best
- USE LOWERCASE
- REMOVE PUNCTUATION
- FILTER POS

In [52]:
# load 5 samples

dummy_set = df.sample(5)

In [59]:
# function for computing similarity between a the false sentence and the options

def compute_similarity(false_sentence, option):
    """compute and similarity scores as scalar between the false sentence and its options"""

    # tokenize
    false_tokens = word_tokenize(false_sentence, preserve_line=True)
    option_tokens = word_tokenize(option, preserve_line=True)

    # compute similarity scores
    scores = []

    for false_token in false_tokens:
        false_synsets = wn.synsets(false_token)

        for option_token in option_tokens:
            option_synsets = wn.synsets(option_token)
            
            for false_synset in false_synsets:
                for option_synset in option_synsets:
                    score = false_synset.wup_similarity(option_synset)
                    if score is not None:
                        scores.append(score)
    if len(scores) == 0:
        return 0.0
            
                
    return sum(scores) / len(scores) 


In [62]:
# find argmin similarity score for each option and list them, compare them to the train_answers.csv
answers = pd.read_csv("train_answers.csv")

correct = 0
total = 0

for index, row in dummy_set.iterrows():

    false_sentence= row["FalseSent"]

    options = {
        "A": row["OptionA"],
        "B": row["OptionB"],
        "C": row["OptionC"]
    }

    #Compute similarity
    scores = {}
    for key, option in options.items():
        scores[key] = compute_similarity(false_sentence, option)

    # argmin 
    predicted = min(scores, key=scores.get)

    # True answers:
    true_answer = answers.loc[index, "answer"]

    # prints
    print(f"\nExample {index}")
    print(f"Scores: {scores}")
    print(f"Predicted: {predicted}")
    print(f"True: {true_answer}")

    if predicted == true_answer:
        correct += 1
        
    total += 1

accuracy = correct / total
print(f"\nAccuracy: {accuracy:.4f}")
    



Example 1392
Scores: {'A': 0.2513471052259256, 'B': 0.260389763080008, 'C': 0.22459610551195067}
Predicted: C
True: A

Example 4774
Scores: {'A': 0.247652625765652, 'B': 0.24762854022416045, 'C': 0.27672973522914435}
Predicted: B
True: A

Example 4774
Scores: {'A': 0.247652625765652, 'B': 0.24762854022416045, 'C': 0.27672973522914435}
Predicted: B
True: A

Example 3816
Scores: {'A': 0.2768603808998943, 'B': 0.27499866014175284, 'C': 0.26278261750536325}
Predicted: C
True: C

Example 3816
Scores: {'A': 0.2768603808998943, 'B': 0.27499866014175284, 'C': 0.26278261750536325}
Predicted: C
True: C

Example 4213
Scores: {'A': 0.22023279761588355, 'B': 0.21841282718030966, 'C': 0.22542305327434636}
Predicted: B
True: C

Example 4213
Scores: {'A': 0.22023279761588355, 'B': 0.21841282718030966, 'C': 0.22542305327434636}
Predicted: B
True: C

Example 6597
Scores: {'A': 0.22947036799317816, 'B': 0.26177505258193795, 'C': 0.2742408203961341}
Predicted: A
True: C

Accuracy: 0.2000

Example 6597
Sc

In [45]:
# run the function on the dummy set

for index, row in dummy_set.iterrows():
    false_sentence = row["FalseSent"]
    options = [row["OptionA"], row["OptionB"], row["OptionC"]]
    print(f"False sentence: {false_sentence}")
    for option in options:
        scores = compute_similarity(false_sentence, option)
        print(f"scores for option {option}: {scores}")

False sentence: A person is not allowed to visit their relatives.
scores for option It is a human right to visit a relative and there are no laws against it, so you can visit a relative.: [0.23529411764705882, 0.1111111111111111, 0.19047619047619047, 0.2, 0.23529411764705882, 0.2222222222222222, 0.21052631578947367, 0.18181818181818182, 0.18181818181818182, 0.18181818181818182, 0.18181818181818182, 0.18181818181818182, 0.18181818181818182, 0.18181818181818182, 0.18181818181818182, 0.1, 0.15384615384615385, 0.18181818181818182, 0.15384615384615385, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.09523809523809523, 0.14285714285714285, 0.16666666666666666, 0.14285714285714285, 0.15384615384615385, 0.14285714285714285, 0.14285714285714285, 0.14285714285714285, 0.14285714285714285, 0.14285714285714285, 0.14285714285714285, 0.14285714285714285, 0.142

# First Test attempt

In [ ]:
# run function on the whole dataset and re

# Tokenizer

In [ ]:
df[]

NameError: name 'token' is not defined

# Vocabulary

In [15]:
from collections import Counter

counter = Counter()

for text in df["FalseSent"]:  # adjust column names
    tokens = simple_tokenizer(text)
    counter.update(tokens)

for word, count in counter.items():
    if count >= 2:
        word_to_idx[word] = len(word_to_idx)

In [20]:
from torch.utils.data import Dataset

class CommonsenseDataset(Dataset):
    def __init__(self, df, vocab, max_len=50):
        self.samples = []
        
        for _, row in df.iterrows():
            text = row["statement"]  # adjust column name
            label = row["label"]     # adjust column name
            
            encoded = encode(text, vocab, max_len)
            self.samples.append((encoded, label))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)

In [17]:
def encode(text, vocab, max_len):
    tokens = simple_tokenizer(text)
    indices = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    
    if len(indices) < max_len:
        indices += [vocab["<PAD>"]] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    
    return indices

In [19]:
def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return text.split()

#config to grab words that appear more than 2 times

word_to_idx = {"<PAD>": 0, "<UNK>": 1}

# idx = word_to_idx.get(token, word_to_idx["<UNK>"])

# if len(seq) < max_len:
#     seq += [0] * (max_len - len(seq))
# else:
#     seq = seq[:max_len]